# LeetCode 1022 — Sum of Root To Leaf Binary Numbers bằng Python

**Link đề:** https://leetcode.com/problems/sum-of-root-to-leaf-binary-numbers/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search (DFS), Binary Tree

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def sumRootToLeaf(self, root: Optional[TreeNode]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu **cây nhị phân**: node, con trái/phải, **lá (leaf)**, đường đi gốc → lá.
2. Tự định nghĩa `TreeNode`, dựng cây từ list kiểu LeetCode (`[1,0,1,null,...]`) và in cây để quan sát.
3. Ôn công thức đọc nhị phân từ trái sang: `x = 2·x + bit` (giống bài 1018).
4. Viết lời giải từ dễ đến gọn: gom mọi đường đi → DFS đệ quy mang giá trị → DFS bằng stack → BFS bằng queue.
5. Trace **call stack** của đệ quy, chứng minh, phân tích Time/Space theo `n` và chiều cao `h`.
6. Làm bài tập biến thể: tổng số thập phân (LeetCode 129), đếm lá, đường đi lớn nhất.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho gốc `root` của một cây nhị phân, mỗi node có giá trị `0` hoặc `1`. Mỗi đường đi từ **gốc tới một lá** biểu diễn một số nhị phân, bit đầu (ở gốc) là bit có trọng số lớn nhất.

Ví dụ đường `0 → 1 → 1 → 0 → 1` biểu diễn `01101₂ = 13`.

Trả về **tổng** các số ứng với mọi lá. Đề đảm bảo đáp án vừa số nguyên 32 bit.

**Ví dụ 1:** `root = [1,0,1,0,1,0,1]`

```
        1
      /   \
     0     1
    / \   / \
   0   1 0   1
```

Bốn đường: `100₂=4`, `101₂=5`, `110₂=6`, `111₂=7` → tổng **22**.

**Ví dụ 2:** `root = [0]` → một node vừa là gốc vừa là lá → `0₂ = 0` → **0**.

**Constraints:** số node trong `[1, 1000]`, `Node.val` là `0` hoặc `1`.

Câu hỏi tự đặt:
- **Lá** là gì? (Node không có con trái **và** không có con phải.)
- Một node chỉ có một con có phải lá không? (Không!)

## 2. Nền tảng: cây nhị phân trong Python

### 2.1. Class `TreeNode`

Một node lưu `val` và hai "con trỏ" `left`, `right` tới node con (hoặc `None` nếu không có).

`class` định nghĩa một kiểu dữ liệu mới; `__init__` là hàm chạy khi tạo đối tượng; `self` là chính đối tượng đó.

In [1]:
from typing import Optional


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


leaf_a = TreeNode(0)
leaf_b = TreeNode(1)
root = TreeNode(1, leaf_a, leaf_b)
print(root, "| trái:", root.left, "| phải:", root.right)
print("leaf_a có con không?", leaf_a.left, leaf_a.right)

TreeNode(1) | trái: TreeNode(0) | phải: TreeNode(1)
leaf_a có con không? None None


- `Optional[TreeNode]` (từ `typing`) nghĩa là "một `TreeNode` hoặc `None`" — dùng trong chữ ký hàm của LeetCode.
- `__repr__` quy định cách đối tượng được in ra — giúp debug dễ hơn.

### 2.2. Dựng cây từ list kiểu LeetCode

LeetCode mô tả cây bằng list theo **thứ tự từng tầng (level order)**, `None` (`null`) là chỗ trống. Ta dựng bằng một **hàng đợi (queue)**: lấy lần lượt từng node, gán cho nó con trái rồi con phải từ list.

`collections.deque` là hàng đợi hai đầu: `append` thêm vào cuối, `popleft` lấy ra từ đầu trong O(1).

In [2]:
from collections import deque


def build_tree(values: list) -> Optional[TreeNode]:
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


q = deque([1, 2])
q.append(3)
print("deque sau append(3):", q)
first = q.popleft()
print("popleft lấy ra:", first, "| còn lại:", q)

deque sau append(3): deque([1, 2, 3])
popleft lấy ra: 1 | còn lại: deque([2, 3])


### 2.3. In cây để nhìn thấy cấu trúc

Hàm `show_tree` in cây nằm ngang (gốc bên trái, con phải ở trên, con trái ở dưới) bằng **đệ quy**: hàm tự gọi lại chính nó cho cây con.

In [3]:
def show_tree(node, indent=0, label="root"):
    if node is None:
        return
    show_tree(node.right, indent + 6, "R")
    print(" " * indent + f"{label}:{node.val}")
    show_tree(node.left, indent + 6, "L")


ex1 = build_tree([1, 0, 1, 0, 1, 0, 1])
show_tree(ex1)

            R:1
      R:1
            L:0
root:1
            R:1
      L:0
            L:0


Đọc hình: xoay đầu sang trái 90° sẽ thấy cây như hình trong đề.

### 2.4. Lá và đệ quy cơ bản

**Đệ quy** gồm: (1) **trường hợp cơ sở** (dừng), (2) **bước đệ quy** (gọi lại trên bài toán nhỏ hơn). Ví dụ đếm số node: cây rỗng có 0 node; ngược lại `1 + đếm(trái) + đếm(phải)`.

In [4]:
def is_leaf(node):
    return node is not None and node.left is None and node.right is None


def count_nodes(node):
    if node is None:            # trường hợp cơ sở
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)


def count_leaves(node):
    if node is None:
        return 0
    if is_leaf(node):
        return 1
    return count_leaves(node.left) + count_leaves(node.right)


print("số node:", count_nodes(ex1), "| số lá:", count_leaves(ex1))

số node: 7 | số lá: 4


### 2.5. Đọc dãy bit thành số: `x = 2·x + bit`

Đi từ gốc xuống là đọc bit từ trái sang phải: thêm bit vào bên phải số cũ = `2·x + bit`.

In [5]:
x = 0
for bit in [0, 1, 1, 0, 1]:
    x = 2 * x + bit
    print(f"thêm {bit} -> x = {x} ({bin(x)[2:]})")
print("int('01101', 2) =", int("01101", 2))

thêm 0 -> x = 0 (0)
thêm 1 -> x = 1 (1)
thêm 1 -> x = 3 (11)
thêm 0 -> x = 6 (110)
thêm 1 -> x = 13 (1101)
int('01101', 2) = 13


## 3. Cách trực giác — gom mọi đường đi rồi đổi sang số

### 3.1. Cách 1: DFS liệt kê đường đi (list các bit), sau đó cộng

Đi sâu từ gốc, giữ list `path` các bit đã qua. Tới lá thì **sao chép** `path` vào danh sách kết quả. Quay lui (backtrack) bằng `path.pop()` khi rời node.

In [6]:
def all_root_to_leaf_paths(root):
    paths = []
    path = []

    def dfs(node):
        if node is None:
            return
        path.append(node.val)
        if is_leaf(node):
            paths.append(path[:])          # sao chép, vì path còn thay đổi
        else:
            dfs(node.left)
            dfs(node.right)
        path.pop()                         # quay lui

    dfs(root)
    return paths


paths = all_root_to_leaf_paths(ex1)
print(paths)

[[1, 0, 0], [1, 0, 1], [1, 1, 0], [1, 1, 1]]


- `def dfs` **bên trong** hàm khác gọi là **hàm lồng (nested function)**; nó đọc/sửa được `path`, `paths` của hàm ngoài (vì ta chỉ gọi `.append`/`.pop`, không gán lại tên biến).
- `path[:]` tạo bản sao; nếu `paths.append(path)` thì mọi phần tử của `paths` cùng trỏ một list và cuối cùng đều rỗng.

In [7]:
def sum_root_to_leaf_paths(root) -> int:
    total = 0
    for p in all_root_to_leaf_paths(root):
        total += int("".join(map(str, p)), 2)
    return total

print(sum_root_to_leaf_paths(ex1))                 # 22
print(sum_root_to_leaf_paths(build_tree([0])))     # 0

22
0


## 4. Nút thắt

Cách 1 đúng, nhưng:
- Mỗi lá phải **sao chép** cả đường đi (độ dài tới `h`) rồi đổi sang số → tổng chi phí O(số lá · h). Với cây "lệch" nhiều nhánh, có thể tới O(n²) / bộ nhớ lớn.
- Ta đang **tính lại** phần đầu chung của nhiều đường (ví dụ `1 → 0` chung cho `100` và `101`).

Insight: giá trị của tiền tố đường đi có thể **mang theo khi đi xuống**. Ở node có giá trị `v`, nếu tiền tố phía trên có giá trị `cur`, thì tiền tố mới là `2·cur + v`. Tới lá thì đó chính là số cần cộng.

In [8]:
# Giá trị tiền tố tại mỗi node của ví dụ 1 (đi theo đường 1 -> 0 -> 1)
cur = 0
for v in [1, 0, 1]:
    cur = 2 * cur + v
    print(f"node {v}: cur = {cur}")

node 1: cur = 1
node 0: cur = 2
node 1: cur = 5


## 5. Lời giải chính — DFS đệ quy mang theo giá trị tiền tố

Hàm `dfs(node, cur)` trả về **tổng các số của mọi lá trong cây con gốc `node`**, biết tiền tố phía trên là `cur`.

```python
def dfs(node, cur):
    if node is None:                 # Đoạn 1: không có cây con -> không đóng góp
        return 0
    cur = 2 * cur + node.val         # Đoạn 2: thêm bit của node
    if node.left is None and node.right is None:
        return cur                   # Đoạn 3: lá -> số hoàn chỉnh
    return dfs(node.left, cur) + dfs(node.right, cur)   # Đoạn 4: cộng hai cây con
```

Vì sao `node is None` trả 0? Khi một node chỉ có một con, nhánh còn lại là `None` — không có lá nào ở đó nên đóng góp 0. Nhờ vậy node có một con **không** bị tính nhầm là lá.

In [9]:
def sum_root_to_leaf_dfs(root) -> int:
    def dfs(node, cur):
        if node is None:
            return 0
        cur = 2 * cur + node.val
        if node.left is None and node.right is None:
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

print(sum_root_to_leaf_dfs(ex1))
print(sum_root_to_leaf_dfs(build_tree([0])))

22
0


### 5.1. Cách 3 — DFS bằng stack (không đệ quy)

Thay call stack bằng stack tự quản: mỗi phần tử là cặp `(node, cur)` — `cur` là giá trị **đã tính tới node đó**.

In [10]:
def sum_root_to_leaf_stack(root) -> int:
    if root is None:
        return 0
    total = 0
    stack = [(root, root.val)]
    while stack:
        node, cur = stack.pop()
        if node.left is None and node.right is None:
            total += cur
            continue
        if node.right:
            stack.append((node.right, 2 * cur + node.right.val))
        if node.left:
            stack.append((node.left, 2 * cur + node.left.val))
    return total

print(sum_root_to_leaf_stack(ex1), sum_root_to_leaf_stack(build_tree([0])))

22 0


- `node, cur = stack.pop()` là **unpacking** cặp (tuple) thành 2 biến.
- `continue` bỏ qua phần còn lại của vòng lặp hiện tại.
- `if node.right:` — node `None` coi là "sai", node thật coi là "đúng".
- Đẩy `right` trước `left` để `left` được lấy ra trước (stack vào sau ra trước) — chỉ ảnh hưởng thứ tự duyệt, không ảnh hưởng tổng.

### 5.2. Cách 4 — BFS theo tầng bằng `deque`

In [11]:
def sum_root_to_leaf_bfs(root) -> int:
    if root is None:
        return 0
    total = 0
    queue = deque([(root, root.val)])
    while queue:
        node, cur = queue.popleft()
        if not node.left and not node.right:
            total += cur
        if node.left:
            queue.append((node.left, 2 * cur + node.left.val))
        if node.right:
            queue.append((node.right, 2 * cur + node.right.val))
    return total

print(sum_root_to_leaf_bfs(ex1), sum_root_to_leaf_bfs(build_tree([0])))

22 0


## 6. TRACE — call stack của DFS đệ quy

Ví dụ 1. Ký hiệu node bằng đường đi từ gốc (`""` là gốc, `"L"`, `"R"`, `"LL"`, ...).

| Lời gọi | node.val | cur vào | cur mới | Lá? | Trả về |
|---|:-:|:-:|:-:|:-:|:-:|
| dfs(gốc, 0) | 1 | 0 | 1 | ✗ | 9 + 13 = 22 |
| └ dfs(L, 1) | 0 | 1 | 2 | ✗ | 4 + 5 = 9 |
| ‧‧└ dfs(LL, 2) | 0 | 2 | 4 | ✓ | 4 |
| ‧‧└ dfs(LR, 2) | 1 | 2 | 5 | ✓ | 5 |
| └ dfs(R, 1) | 1 | 1 | 3 | ✗ | 6 + 7 = 13 |
| ‧‧└ dfs(RL, 3) | 0 | 3 | 6 | ✓ | 6 |
| ‧‧└ dfs(RR, 3) | 1 | 3 | 7 | ✓ | 7 |

Giá trị đi **xuống** qua tham số `cur`; tổng đi **lên** qua giá trị trả về.

In [12]:
def trace_dfs(root):
    def dfs(node, cur, name, depth):
        pad = "    " * depth
        if node is None:
            print(f"{pad}dfs({name or 'gốc'}): None -> 0")
            return 0
        new_cur = 2 * cur + node.val
        if node.left is None and node.right is None:
            print(f"{pad}dfs({name or 'gốc'}, cur={cur}): val={node.val} -> cur={new_cur}, LÁ -> trả {new_cur}")
            return new_cur
        print(f"{pad}dfs({name or 'gốc'}, cur={cur}): val={node.val} -> cur={new_cur}, đi xuống")
        left = dfs(node.left, new_cur, name + "L", depth + 1)
        right = dfs(node.right, new_cur, name + "R", depth + 1)
        print(f"{pad}<- {name or 'gốc'} trả {left} + {right} = {left + right}")
        return left + right
    return dfs(root, 0, "", 0)

print("Tổng:", trace_dfs(ex1))

dfs(gốc, cur=0): val=1 -> cur=1, đi xuống
    dfs(L, cur=1): val=0 -> cur=2, đi xuống
        dfs(LL, cur=2): val=0 -> cur=4, LÁ -> trả 4
        dfs(LR, cur=2): val=1 -> cur=5, LÁ -> trả 5
    <- L trả 4 + 5 = 9
    dfs(R, cur=1): val=1 -> cur=3, đi xuống
        dfs(RL, cur=3): val=0 -> cur=6, LÁ -> trả 6
        dfs(RR, cur=3): val=1 -> cur=7, LÁ -> trả 7
    <- R trả 6 + 7 = 13
<- gốc trả 9 + 13 = 22
Tổng: 22


### 6.1. Trace một cây có node chỉ một con

Cây `[1,1,null,0,1]`:

```
      1
     /
    1
   / \
  0   1
```

Node gốc chỉ có con trái → không phải lá. Nhánh phải `None` đóng góp 0. Các đường: `110₂ = 6`, `111₂ = 7` → tổng 13.

In [13]:
t2 = build_tree([1, 1, None, 0, 1])
show_tree(t2)
print("Tổng:", trace_dfs(t2))

root:1
            R:1
      L:1
            L:0
dfs(gốc, cur=0): val=1 -> cur=1, đi xuống
    dfs(L, cur=1): val=1 -> cur=3, đi xuống
        dfs(LL, cur=3): val=0 -> cur=6, LÁ -> trả 6
        dfs(LR, cur=3): val=1 -> cur=7, LÁ -> trả 7
    <- L trả 6 + 7 = 13
    dfs(R): None -> 0
<- gốc trả 13 + 0 = 13
Tổng: 13


## 7. Code nộp LeetCode (Python 3)

LeetCode đã định nghĩa sẵn `TreeNode` và `Optional`; trong notebook ta dùng class tự định nghĩa ở mục 2.1.

In [14]:
class Solution:
    def sumRootToLeaf(self, root: Optional[TreeNode]) -> int:
        def dfs(node, cur):
            if node is None:
                return 0
            cur = 2 * cur + node.val
            if node.left is None and node.right is None:
                return cur
            return dfs(node.left, cur) + dfs(node.right, cur)

        return dfs(root, 0)

sol = Solution()
assert sol.sumRootToLeaf(build_tree([1, 0, 1, 0, 1, 0, 1])) == 22
assert sol.sumRootToLeaf(build_tree([0])) == 0
print("Solution PASS 2 ví dụ của đề")

Solution PASS 2 ví dụ của đề


Phiên bản lặp (an toàn với cây rất sâu, không lo giới hạn đệ quy):

In [15]:
class SolutionIterative:
    def sumRootToLeaf(self, root: Optional[TreeNode]) -> int:
        total = 0
        stack = [(root, 0)]
        while stack:
            node, cur = stack.pop()
            if node is None:
                continue
            cur = 2 * cur + node.val
            if node.left is None and node.right is None:
                total += cur
            else:
                stack.append((node.left, cur))
                stack.append((node.right, cur))
        return total

assert SolutionIterative().sumRootToLeaf(build_tree([1, 0, 1, 0, 1, 0, 1])) == 22
assert SolutionIterative().sumRootToLeaf(build_tree([1, 1, None, 0, 1])) == 13
print("SolutionIterative PASS")

SolutionIterative PASS


## 8. Vì sao đúng?

**Bất biến:** khi gọi `dfs(node, cur)`, `cur` là giá trị nhị phân của đường đi từ gốc tới **cha** của `node` (với gốc: `cur = 0`, đường rỗng).

- Sau `cur = 2·cur + node.val`, `cur` là giá trị đường đi từ gốc tới chính `node` (thêm bit vào bên phải — mục 2.5).
- Nếu `node` là lá: `cur` chính là số của đường gốc → lá đó. Trả về nó.
- Nếu không: mọi lá trong cây con gốc `node` nằm ở **đúng một** trong hai cây con trái/phải → tổng = tổng bên trái + tổng bên phải. Nhánh `None` không có lá → 0.

Quy nạp theo chiều cao cây con ⇒ `dfs(root, 0)` = tổng số của mọi lá. Mỗi lá được đếm **đúng một lần** vì mỗi lá có duy nhất một đường từ gốc.

**Về tràn số:** đề đảm bảo đáp án vừa 32 bit; Python có số nguyên không giới hạn nên không lo tràn trong quá trình tính.

## 9. Phân tích Time / Space

Gọi `n` = số node, `h` = chiều cao cây (số node trên đường dài nhất), `L` = số lá.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm |
|---|---|---|---|
| 1. Gom đường đi | sao chép path ở mỗi lá | `O(n + L·h)` | `O(L·h)` |
| 2. DFS đệ quy mang `cur` | mỗi node thăm 1 lần | `O(n)` | `O(h)` call stack |
| 3. DFS bằng stack | như trên, stack tự quản | `O(n)` | `O(h)` (≈, tối đa `O(n)`) |
| 4. BFS bằng queue | duyệt theo tầng | `O(n)` | `O(w)`, `w` = độ rộng tầng lớn nhất |

- Cây cân bằng: `h ≈ log₂ n`. Cây lệch (như danh sách): `h = n`.
- Với `n ≤ 1000`, đệ quy sâu tối đa 1000 — vừa sát giới hạn mặc định của Python (`sys.getrecursionlimit()` thường là 1000). LeetCode nâng giới hạn này, nhưng cách lặp là an toàn tuyệt đối.

In [16]:
import sys
print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

Giới hạn đệ quy hiện tại: 1000


## 10. Kiểm thử: mẫu đề, biên, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<36} PASS ({actual})")

cases = [
    ("đề 1", [1, 0, 1, 0, 1, 0, 1], 22),
    ("đề 2", [0], 0),
    ("một node 1", [1], 1),
    ("gốc chỉ có con trái", [1, 1, None, 0, 1], 13),
    ("chuỗi lệch phải 1->1->1", [1, None, 1, None, 1], 7),
    ("gốc 0, hai lá", [0, 1, 1], 2),
    ("lá ở độ sâu khác nhau", [1, 0, 1, None, None, 1, 0], 2 + 7 + 6),
]
solvers = {
    "paths": sum_root_to_leaf_paths,
    "dfs": sum_root_to_leaf_dfs,
    "stack": sum_root_to_leaf_stack,
    "bfs": sum_root_to_leaf_bfs,
    "Solution": Solution().sumRootToLeaf,
    "SolutionIterative": SolutionIterative().sumRootToLeaf,
}
for name, values, exp in cases:
    root = build_tree(values)
    for sname, f in solvers.items():
        assert f(root) == exp, (name, sname)
    check(name, Solution().sumRootToLeaf(root), exp)

đề 1                                 PASS (22)
đề 2                                 PASS (0)
một node 1                           PASS (1)
gốc chỉ có con trái                  PASS (13)
chuỗi lệch phải 1->1->1              PASS (7)
gốc 0, hai lá                        PASS (2)
lá ở độ sâu khác nhau                PASS (15)


### 10.1. Cây ngẫu nhiên (có seed) — đối chiếu mọi cách

In [18]:
import random
random.seed(1022)

def random_tree(n_nodes: int) -> Optional[TreeNode]:
    # thêm lần lượt n_nodes node vào vị trí trống ngẫu nhiên
    root = TreeNode(random.randint(0, 1))
    nodes = [root]
    for _ in range(n_nodes - 1):
        while True:
            parent = random.choice(nodes)
            side = random.choice(["left", "right"])
            if getattr(parent, side) is None:
                child = TreeNode(random.randint(0, 1))
                setattr(parent, side, child)
                nodes.append(child)
                break
    return root

for _ in range(500):
    root = random_tree(random.randint(1, 30))
    expected = sum_root_to_leaf_paths(root)
    for sname, f in solvers.items():
        assert f(root) == expected, sname
print("500 cây ngẫu nhiên: mọi cách khớp nhau")

500 cây ngẫu nhiên: mọi cách khớp nhau


`getattr(obj, "left")` đọc thuộc tính theo **tên dạng chuỗi**; `setattr(obj, "left", v)` gán thuộc tính theo tên. Giúp chọn trái/phải ngẫu nhiên mà không viết `if`.

### 10.2. Cây lệch 1000 node — đệ quy vs lặp

In [19]:
def chain_tree(n):
    root = TreeNode(1)
    cur = root
    for _ in range(n - 1):
        cur.left = TreeNode(0)
        cur = cur.left
    return root

deep = chain_tree(1000)
print("lặp:", SolutionIterative().sumRootToLeaf(deep) == 2 ** 999)
try:
    Solution().sumRootToLeaf(deep)
    print("đệ quy: chạy được")
except RecursionError:
    print("đệ quy: RecursionError (vượt giới hạn mặc định) -> dùng bản lặp hoặc sys.setrecursionlimit")

lặp: True
đệ quy: RecursionError (vượt giới hạn mặc định) -> dùng bản lặp hoặc sys.setrecursionlimit


Đường `1 → 0 → 0 → ... → 0` (1000 node) là `2^999` — rất lớn (đề thật đảm bảo đáp án vừa 32 bit nên sẽ không có cây như vậy, nhưng nó giúp kiểm tra **độ sâu**). Python vẫn tính chính xác nhờ số nguyên không giới hạn.

## 11. Sai lầm người mới hay mắc

### 11.1. Cộng tại node `None` thay vì tại lá

In [20]:
def wrong_add_at_none(root):
    def dfs(node, cur):
        if node is None:
            return cur                       # SAI: mỗi nhánh None đều cộng một lần
        cur = 2 * cur + node.val
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

print("sai :", wrong_add_at_none(build_tree([1, 0, 1, 0, 1, 0, 1])))
print("đúng:", Solution().sumRootToLeaf(build_tree([1, 0, 1, 0, 1, 0, 1])))

sai : 44
đúng: 22


**Vì sao sai:** mỗi lá có **hai** con `None` → số của lá bị cộng 2 lần (44 thay vì 22); node một con còn bị cộng thêm qua nhánh `None` của nó. **Sửa:** chỉ cộng khi node là lá thật (`left is None and right is None`), `None` trả 0.

### 11.2. Coi node có một con là lá

In [21]:
def wrong_leaf_check(root):
    def dfs(node, cur):
        if node is None:
            return 0
        cur = 2 * cur + node.val
        if node.left is None or node.right is None:     # SAI: 'or'
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

t = build_tree([1, 1, None, 0, 1])
print("sai :", wrong_leaf_check(t))
print("đúng:", Solution().sumRootToLeaf(t))

sai : 1
đúng: 13


**Vì sao sai:** gốc chỉ có con trái → `or` làm gốc bị coi là lá, trả 1 và bỏ qua cả cây con. **Sửa:** `and`.

### 11.3. Dùng biến toàn cục `cur` mà không khôi phục khi quay lui

In [22]:
def wrong_shared_cur(root):
    state = {"cur": 0, "total": 0}
    def dfs(node):
        if node is None:
            return
        state["cur"] = 2 * state["cur"] + node.val     # SAI: không hoàn tác khi quay lui
        if node.left is None and node.right is None:
            state["total"] += state["cur"]
        dfs(node.left)
        dfs(node.right)
    dfs(root)
    return state["total"]

print("sai :", wrong_shared_cur(build_tree([1, 0, 1, 0, 1, 0, 1])))
print("đúng:", Solution().sumRootToLeaf(build_tree([1, 0, 1, 0, 1, 0, 1])))

sai : 128
đúng: 22


**Vì sao sai:** sau khi đi xuống nhánh trái, `cur` vẫn mang bit của nhánh trái khi sang nhánh phải. **Sửa:** truyền `cur` qua **tham số** (mỗi lời gọi có bản riêng), hoặc hoàn tác `cur = (cur - node.val) // 2` trước khi `return`.

### 11.4. Đọc bit theo chiều ngược (lá là bit cao nhất)

Nếu cộng `node.val << depth` thì gốc thành bit thấp nhất — ngược đề. Đường `1 → 0 → 0` phải là `100₂ = 4`, không phải `001₂ = 1`. **Sửa:** dùng `cur = 2·cur + val` khi đi xuống.

## 12. Nhận diện pattern

Dấu hiệu:
- "đường đi từ gốc tới lá" + "giá trị tích lũy" → **DFS mang trạng thái đi xuống** qua tham số.
- "tổng/đếm trên mọi lá" → DFS trả giá trị đi lên và cộng hai cây con.
- Trạng thái chỉ phụ thuộc tiền tố đường đi (`2·cur + val`, `10·cur + val`, `cur + val`) → không cần lưu cả đường.

Bài liên quan:
- LeetCode 129 (Sum Root to Leaf Numbers) — y hệt nhưng hệ 10: `10·cur + val`.
- LeetCode 112 / 113 (Path Sum I/II) — tổng dọc đường, so với target.
- LeetCode 257 (Binary Tree Paths) — liệt kê đường đi (giống Cách 1).
- LeetCode 1290 (Convert Binary Number in a Linked List to Integer) — cùng công thức trên danh sách liên kết.

## 13. Biến thể

### 13.1. Biến thể A — LeetCode 129: tổng các số **thập phân** gốc → lá

In [23]:
def sum_numbers_base10(root) -> int:
    def dfs(node, cur):
        if node is None:
            return 0
        cur = 10 * cur + node.val
        if node.left is None and node.right is None:
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)

print(sum_numbers_base10(build_tree([1, 2, 3])))          # 12 + 13 = 25
print(sum_numbers_base10(build_tree([4, 9, 0, 5, 1])))    # 495 + 491 + 40 = 1026

25
1026


### 13.2. Biến thể B — tổng theo **cơ số bất kỳ** và trả về cả danh sách số của từng lá

In [24]:
def leaf_numbers(root, base=2) -> list[int]:
    out = []
    def dfs(node, cur):
        if node is None:
            return
        cur = base * cur + node.val
        if node.left is None and node.right is None:
            out.append(cur)
            return
        dfs(node.left, cur)
        dfs(node.right, cur)
    dfs(root, 0)
    return out

print(leaf_numbers(ex1))            # [4, 5, 6, 7]
print(leaf_numbers(ex1, base=10))   # [100, 101, 110, 111]

[4, 5, 6, 7]
[100, 101, 110, 111]


## 14. Hệ thống luyện tập — 6 bài

### Bài 1 — Tính tay
Cho cây `[1,0,0,1,null,1,1]`. Vẽ cây, liệt kê các đường gốc → lá và tính tổng.

### Bài 2 — Dự đoán output
Đoán `sum_root_to_leaf_dfs(build_tree([1, None, 0, None, 1]))` và `count_leaves(build_tree([1, 0, 1, 0, 1, 0, 1]))`.

### Bài 3 — TODO: viết lại DFS đệ quy mang `cur`

In [25]:
def exercise_sum_root_to_leaf(root: Optional[TreeNode]) -> int:
    # TODO 1: hàm dfs(node, cur): None -> 0
    # TODO 2: cur = 2*cur + node.val
    # TODO 3: lá -> trả cur; không -> cộng hai nhánh
    pass

# assert exercise_sum_root_to_leaf(build_tree([1, 0, 1, 0, 1, 0, 1])) == 22
# assert exercise_sum_root_to_leaf(build_tree([1, 1, None, 0, 1])) == 13

### Bài 4 — TODO: viết bằng BFS (queue), không đệ quy

In [26]:
def exercise_sum_bfs(root: Optional[TreeNode]) -> int:
    # TODO: deque các cặp (node, cur)
    pass

# assert exercise_sum_bfs(build_tree([1, 0, 1, 0, 1, 0, 1])) == 22
# assert exercise_sum_bfs(build_tree([0])) == 0

### Bài 5 — TODO biến thể: số **lớn nhất** trong các số gốc → lá

In [27]:
def exercise_max_leaf_number(root: Optional[TreeNode]) -> int:
    # TODO
    pass

# assert exercise_max_leaf_number(build_tree([1, 0, 1, 0, 1, 0, 1])) == 7
# assert exercise_max_leaf_number(build_tree([1, 1, None, 0, 1])) == 7

### Bài 6 — Mini challenge: trả về đường đi (list giá trị) của lá có số lớn nhất

Nếu nhiều lá cùng số lớn nhất, trả đường của lá gặp trước theo thứ tự trái → phải.

In [28]:
def exercise_max_leaf_path(root: Optional[TreeNode]) -> list[int]:
    # TODO
    pass

# assert exercise_max_leaf_path(build_tree([1, 0, 1, 0, 1, 0, 1])) == [1, 1, 1]
# assert exercise_max_leaf_path(build_tree([1, 1, 0, 0, None, 1, 1])) == [1, 1, 0]

## 15. Gợi ý ba mức

**Bài 1** — (1) dựng bằng tay theo tầng: gốc 1, tầng 2: `0, 0`, tầng 3: con trái của node 0 thứ nhất là `1`, con phải trống, rồi `1, 1`; (2) có 3 lá; (3) các số có 3 bit.

**Bài 2** — (1) cây lệch phải `1 → 0 → 1`; (2) một lá duy nhất; (3) ví dụ 1 có 4 lá.

**Bài 3** — (1) mục 5; (2) điều kiện lá dùng `and`; (3) `return dfs(node.left, cur) + dfs(node.right, cur)`.

**Bài 4** — (1) mục 5.2; (2) `queue.popleft()`; (3) khi đẩy con: `2 * cur + child.val`.

**Bài 5** — (1) giống bài 3 nhưng thay `+` bằng `max`; (2) nhánh `None` phải trả giá trị **không ảnh hưởng** `max` (ví dụ `-1`); (3) lá trả `cur`.

**Bài 6** — (1) giữ `path` và `best = (giá trị, đường)`; (2) chỉ cập nhật khi `cur > best_val` (lớn hơn **thật sự** để giữ lá đầu tiên); (3) duyệt trái trước phải, nhớ `path.pop()`.

## 16. Đáp án tham khảo

**Bài 1:** cây `[1,0,0,1,null,1,1]`:

```
        1
      /   \
     0     0
    /     / \
   1     1   1
```
Đường: `101₂=5`, `101₂=5`, `101₂=5` → tổng **15**.

**Bài 2:** `101₂ = 5`; ví dụ 1 có `4` lá.

In [29]:
def answer_sum_root_to_leaf(root):
    def dfs(node, cur):
        if node is None:
            return 0
        cur = 2 * cur + node.val
        if node.left is None and node.right is None:
            return cur
        return dfs(node.left, cur) + dfs(node.right, cur)
    return dfs(root, 0)


def answer_sum_bfs(root):
    if root is None:
        return 0
    total = 0
    queue = deque([(root, root.val)])
    while queue:
        node, cur = queue.popleft()
        if node.left is None and node.right is None:
            total += cur
        for child in (node.left, node.right):
            if child is not None:
                queue.append((child, 2 * cur + child.val))
    return total


def answer_max_leaf_number(root):
    def dfs(node, cur):
        if node is None:
            return -1
        cur = 2 * cur + node.val
        if node.left is None and node.right is None:
            return cur
        return max(dfs(node.left, cur), dfs(node.right, cur))
    return dfs(root, 0)


def answer_max_leaf_path(root):
    best = {"val": -1, "path": []}
    path = []

    def dfs(node, cur):
        if node is None:
            return
        cur = 2 * cur + node.val
        path.append(node.val)
        if node.left is None and node.right is None:
            if cur > best["val"]:
                best["val"] = cur
                best["path"] = path[:]
        else:
            dfs(node.left, cur)
            dfs(node.right, cur)
        path.pop()

    dfs(root, 0)
    return best["path"]


assert answer_sum_root_to_leaf(build_tree([1, 0, 0, 1, None, 1, 1])) == 15
assert sum_root_to_leaf_dfs(build_tree([1, None, 0, None, 1])) == 5
assert count_leaves(build_tree([1, 0, 1, 0, 1, 0, 1])) == 4
assert answer_sum_root_to_leaf(build_tree([1, 1, None, 0, 1])) == 13
assert answer_sum_bfs(build_tree([1, 0, 1, 0, 1, 0, 1])) == 22 and answer_sum_bfs(build_tree([0])) == 0
assert answer_max_leaf_number(build_tree([1, 0, 1, 0, 1, 0, 1])) == 7
assert answer_max_leaf_number(build_tree([1, 1, None, 0, 1])) == 7
assert answer_max_leaf_path(build_tree([1, 0, 1, 0, 1, 0, 1])) == [1, 1, 1]
assert answer_max_leaf_path(build_tree([1, 1, 0, 0, None, 1, 1])) == [1, 1, 0]

random.seed(99)
for _ in range(300):
    root = random_tree(random.randint(1, 25))
    nums = leaf_numbers(root)
    assert answer_sum_bfs(root) == sum(nums)
    assert answer_max_leaf_number(root) == max(nums)
    p = answer_max_leaf_path(root)
    assert int("".join(map(str, p)), 2) == max(nums)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


**Giải thích bài 6:** `best` là `dict` để hàm lồng có thể **gán lại** giá trị bên trong (nếu viết `best_val = cur` trong hàm lồng, Python sẽ tạo biến cục bộ mới; muốn gán biến của hàm ngoài phải dùng `nonlocal best_val`). Dùng `>` (không phải `>=`) để giữ lá gặp **đầu tiên**.

Lưu ý: đường đi ngắn hơn có thể có giá trị nhỏ hơn dù toàn bit 1 (`11₂ = 3 < 100₂ = 4`), nên phải so sánh **giá trị** chứ không so sánh list.

## 17. Tổng kết — template + checklist

**Template nhớ nhanh (DFS mang trạng thái đi xuống, tổng đi lên):**

```python
class Solution:
    def sumRootToLeaf(self, root: Optional[TreeNode]) -> int:
        def dfs(node, cur):
            if node is None:
                return 0
            cur = 2 * cur + node.val
            if node.left is None and node.right is None:
                return cur
            return dfs(node.left, cur) + dfs(node.right, cur)
        return dfs(root, 0)
```

**Câu thần chú:** *"Xuống: nhân hai cộng bit — Lá: trả số — None: trả 0 — Lên: cộng hai nhánh."*

**Checklist trước khi submit:**
- [ ] Lá = `left is None and right is None` (dùng `and`)?
- [ ] Node `None` trả `0` (không cộng `cur` ở đó)?
- [ ] `cur` truyền qua tham số (không dùng biến chung mà quên hoàn tác)?
- [ ] Công thức `2 * cur + val` (gốc là bit cao nhất)?
- [ ] Đã test cây một node, cây có node một con, cây lệch?
- [ ] Nếu cây có thể rất sâu: dùng bản lặp?